# 02. Conservative CatBoost — promo1

- 데이터: `PUBLIC/FINAL_promo_1.csv`
- 모델: CatBoostClassifier 보수형 탐색
- 목적: 기존 CatBoost trial pool의 높은 과적합 비율을 줄이기 위한 보수형 재실행
- HPT: Optuna `n_trials=100`, 최적화 기준 = `objective_value = mean_valid_auc - GAP_PENALTY * max(0, gap)`
- CV: StratifiedKFold `n_splits=5`, `random_state=42`
- 과적합 기준: `train_auc - valid_auc > 0.03` → `overfit=True`
- 최적 파라미터 선택: `overfit=False` 중 `objective_value` 최고, 동률 시 `mean_valid_auc` 최고
- 보수 제약: 낮은 depth, 낮은 learning_rate, 높은 l2_leaf_reg, early stopping, min_data_in_leaf 적용
- 평가 지표: ROC-AUC, PR-AUC, F1, Precision, Recall
- 산출물: `trials_all.csv`, `final_result.csv`

주의: 이 노트북은 SHAP, segmentation, row-level score table을 생성하지 않는다.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
)
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_TRIALS     = 100
N_SPLITS     = 5
OVERFIT_GAP  = 0.03
GAP_PENALTY  = 0.50
TEST_SIZE    = 0.2
PROMO        = 1

ROOT    = Path(r"C:\Users\USER\OneDrive\바탕 화면\AX git\ott-churn-prediction\PUBLIC")
DATA    = ROOT / "FINAL_promo_1.csv"
OUT_DIR = ROOT / "results" / "02_catboost_promo1_conservative"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("출력 폴더:", OUT_DIR)


출력 폴더: c:\Code\ott-churn-prediction\PUBLIC\results\02_catboost_promo1_conservative


In [2]:
df = pd.read_csv(DATA)
print(f"데이터 shape: {df.shape}")
print(f"is_repurchase 분포:\n{df['is_repurchase'].value_counts()}")

TARGET    = 'is_repurchase'
DROP_COLS = ['USER_KEY', TARGET]
FEATURES  = [c for c in df.columns if c not in DROP_COLS]

X = df[FEATURES].values
y = df[TARGET].values
print(f"\n피처 수: {len(FEATURES)}")
print(f"샘플 수: {len(y)}")

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"\ntrain_valid: {X_tv.shape[0]}행  |  test: {X_test.shape[0]}행")
print(f"train_valid 양성률: {y_tv.mean():.4f}")
print(f"test 양성률:        {y_test.mean():.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Code\\ott-churn-prediction\\PUBLIC\\FINAL_promo_1.csv'

In [ ]:
def build_model(trial):
    """Conservative CatBoost search space.

    보수 설계 원칙:
    - depth를 낮게 제한해 복잡한 상호작용 암기를 줄인다.
    - learning_rate를 낮게 제한한다.
    - l2_leaf_reg를 높게 잡아 leaf value 과적합을 억제한다.
    - random_strength, bagging_temperature로 규제/무작위성을 부여한다.
    - min_data_in_leaf를 키워 작은 패턴 암기를 줄인다.
    - objective 단계에서 eval_set 기반 early stopping을 적용한다.
    """
    params = {
        "iterations":        trial.suggest_int("iterations", 300, 1200),
        "depth":             trial.suggest_int("depth", 3, 5),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 10.0, 100.0, log=True),
        "random_strength":   trial.suggest_float("random_strength", 1.0, 20.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count":      trial.suggest_int("border_count", 32, 128),
        "min_data_in_leaf":  trial.suggest_int("min_data_in_leaf", 20, 100),
    }
    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        od_type="Iter",
        od_wait=50,
        use_best_model=True,
    )
    return model


In [ ]:
def objective(trial):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    train_aucs, valid_aucs = [], []

    for tr_idx, va_idx in skf.split(X_tv, y_tv):
        X_tr, X_va = X_tv[tr_idx], X_tv[va_idx]
        y_tr, y_va = y_tv[tr_idx], y_tv[va_idx]

        model = build_model(trial)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, verbose=False)

        train_aucs.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1]))
        valid_aucs.append(roc_auc_score(y_va, model.predict_proba(X_va)[:, 1]))

    mean_train = float(np.mean(train_aucs))
    mean_valid = float(np.mean(valid_aucs))
    gap        = mean_train - mean_valid
    overfit    = gap > OVERFIT_GAP

    # 보수형 목적함수:
    # mean_valid_auc만 최대화하지 않고 train-valid gap에 벌점을 준다.
    objective_value = mean_valid - GAP_PENALTY * max(0.0, gap)

    trial.set_user_attr('mean_train_auc', mean_train)
    trial.set_user_attr('mean_valid_auc', mean_valid)
    trial.set_user_attr('gap',            gap)
    trial.set_user_attr('overfit',        overfit)
    trial.set_user_attr('objective_value', objective_value)
    return objective_value

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Optuna 완료")


In [ ]:
rows = []
for t in study.trials:
    row = {
        'trial':           t.number,
        'objective_value': t.user_attrs.get('objective_value', float('nan')),
        'mean_valid_auc':  t.user_attrs.get('mean_valid_auc',  float('nan')),
        'mean_train_auc':  t.user_attrs.get('mean_train_auc',  float('nan')),
        'gap':             t.user_attrs.get('gap',             float('nan')),
        'overfit':         t.user_attrs.get('overfit',         float('nan')),
    }
    row.update(t.params)
    rows.append(row)

trials_df = pd.DataFrame(rows).sort_values(
    ['objective_value', 'mean_valid_auc'],
    ascending=[False, False]
)
trials_df.to_csv(OUT_DIR / 'trials_all.csv', index=False, encoding='utf-8-sig')
print(f"전체 trials: {len(trials_df)}개")
print(f"과적합(gap>0.03): {int(trials_df['overfit'].sum())}개")
print("
상위 10 trials:")
print(trials_df.head(10)[['trial','objective_value','mean_valid_auc','mean_train_auc','gap','overfit']].to_string(index=False))


In [ ]:
non_overfit = trials_df[trials_df['overfit'] == False]
if len(non_overfit) == 0:
    print('⚠️  과적합이 아닌 trial 없음 → objective_value 기준 전체 최고 trial 선택')
    best_row = trials_df.iloc[0]
else:
    best_row = non_overfit.sort_values(
        ['objective_value', 'mean_valid_auc'],
        ascending=[False, False]
    ).iloc[0]

param_cols  = [c for c in trials_df.columns if c not in [
    'trial', 'objective_value', 'mean_valid_auc',
    'mean_train_auc', 'gap', 'overfit'
]]
best_params = best_row[param_cols].to_dict()

print(f"
최적 trial: {int(best_row['trial'])}")
print(f"objective_value: {best_row['objective_value']:.4f}")
print(f"mean_valid_auc:  {best_row['mean_valid_auc']:.4f}")
print(f"gap: {best_row['gap']:.4f}  |  overfit: {best_row['overfit']}")
print('파라미터:', best_params)


In [ ]:
class _FakeTrial:
    def __init__(self, p): self._p = p
    def suggest_int(self, n, *a, **k):         return int(self._p[n])
    def suggest_float(self, n, *a, **k):       return float(self._p[n])
    def suggest_categorical(self, n, *a, **k): return self._p[n]

# 최종 모델 학습 시에도 train_valid 내부 validation split을 만들어 early stopping을 적용한다.
# test set은 마지막 평가에만 사용한다.
X_fit, X_es, y_fit, y_es = train_test_split(
    X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=RANDOM_STATE
)

final_model = build_model(_FakeTrial(best_params))
final_model.fit(X_fit, y_fit, eval_set=(X_es, y_es), use_best_model=True, verbose=False)

p_train = final_model.predict_proba(X_fit)[:, 1]
p_test  = final_model.predict_proba(X_test)[:, 1]
y_pred  = (p_test >= 0.5).astype(int)

final_train_auc = float(roc_auc_score(y_fit, p_train))
final_test_auc  = float(roc_auc_score(y_test, p_test))
final_gap_proxy = final_train_auc - final_test_auc

final_result = {
    'model':                 'CatBoost_conservative',
    'promo':                 1,
    'data_file':             'FINAL_promo_1.csv',
    'n_features':            int(len(FEATURES)),
    'feature_names':         '|'.join(FEATURES),
    'n_total':               int(len(y)),
    'n_train_valid':         int(len(y_tv)),
    'n_test':                int(len(y_test)),
    'random_state':          RANDOM_STATE,
    'n_trials':              N_TRIALS,
    'n_splits':              N_SPLITS,
    'overfit_gap':           OVERFIT_GAP,
    'gap_penalty':           GAP_PENALTY,
    'best_trial':            int(best_row['trial']),
    'best_objective_value':  float(best_row['objective_value']),
    'best_valid_auc':        float(best_row['mean_valid_auc']),
    'best_train_auc':        float(best_row['mean_train_auc']),
    'best_gap':              float(best_row['gap']),
    'overfit':               bool(best_row['overfit']),
    'final_train_auc':       final_train_auc,
    'final_gap_proxy':       final_gap_proxy,
    'test_roc_auc':          final_test_auc,
    'test_pr_auc':           float(average_precision_score(y_test, p_test)),
    'test_f1':               float(f1_score(y_test, y_pred)),
    'test_precision':        float(precision_score(y_test, y_pred)),
    'test_recall':           float(recall_score(y_test, y_pred)),
    **{f'param_{k}': v for k, v in best_params.items()},
}

pd.DataFrame([final_result]).to_csv(OUT_DIR / 'final_result.csv', index=False, encoding='utf-8-sig')
print(pd.DataFrame([final_result]).T)
print("
저장 완료:")
print("-", OUT_DIR / 'trials_all.csv')
print("-", OUT_DIR / 'final_result.csv')
